# OpenTelemetry Multi-Service Trace Simulation

This notebook simulates a system with four services:
- **Frontend** → **Gateway** → **Backend** → **LLM Service**

Each service instruments its work as a child span within the same end-to-end trace, identified by a `session.id` attribute. Traces are exported to a local OpenTelemetry Collector (default: `localhost:4317` via OTLP/gRPC).

## 1. Install Dependencies

Dependencies are managed by **uv** in the `.venv` in this directory.  \nSelect the **`otel-experiment`** kernel in Jupyter before running.

To add a new package:
```bash
uv add <package-name>
```

## 2. Configure the Tracer

Set up the OTLP exporter pointing at your local collector and create a shared `TracerProvider`.

In [1]:
import uuid
import time

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.trace.propagation.tracecontext import TraceContextTextMapPropagator
from opentelemetry import context, baggage
from opentelemetry.propagators.composite import CompositePropagator
from opentelemetry.baggage.propagation import W3CBaggagePropagator

# ── Configuration ────────────────────────────────────────────────────────────
COLLECTOR_ENDPOINT = "http://localhost:4317"   # change if your collector listens elsewhere
INSECURE = True                                # set False if TLS is enabled on the collector

# ── Shared resource tags (appear on every span) ───────────────────────────────
resource = Resource.create({
    "deployment.environment": "notebook-demo",
})

# ── Exporters ────────────────────────────────────────────────────────────────
otlp_exporter = OTLPSpanExporter(
    endpoint=COLLECTOR_ENDPOINT,
    insecure=INSECURE,
)
console_exporter = ConsoleSpanExporter()       # prints spans locally so you can verify without a collector

# ── Provider ─────────────────────────────────────────────────────────────────
provider = TracerProvider(resource=resource)
provider.add_span_processor(BatchSpanProcessor(otlp_exporter))
provider.add_span_processor(BatchSpanProcessor(console_exporter))

trace.set_tracer_provider(provider)

propagator = CompositePropagator([TraceContextTextMapPropagator(), W3CBaggagePropagator()])

print("TracerProvider configured — exporting to", COLLECTOR_ENDPOINT)

TracerProvider configured — exporting to http://localhost:4317


In [2]:

# ═════════════════════════════════════════════════════════════════════════════
# OTEL Metrics Setup for LLM Token Streaming (TTFT & TPS)
# ═════════════════════════════════════════════════════════════════════════════

from opentelemetry import metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.exporter.otlp.proto.grpc.metric_exporter import OTLPMetricExporter

# ── Metrics Exporter ──────────────────────────────────────────────────────────
otlp_metric_exporter = OTLPMetricExporter(
    endpoint=COLLECTOR_ENDPOINT,
    insecure=INSECURE,
)

# ── Metrics Provider ──────────────────────────────────────────────────────────
metric_reader = PeriodicExportingMetricReader(otlp_metric_exporter)
metric_provider = MeterProvider(resource=resource, metric_readers=[metric_reader])
metrics.set_meter_provider(metric_provider)

meter = metrics.get_meter("llm-service")

# ── LLM Metrics ───────────────────────────────────────────────────────────────
# Histogram for Time to First Token (milliseconds)
ttft_histogram = meter.create_histogram(
    name="llm.ttft",
    description="Time to First Token in milliseconds",
    unit="ms",
)

# Histogram for Tokens Per Second throughput
tps_histogram = meter.create_histogram(
    name="llm.tokens_per_second",
    description="Tokens per second throughput",
    unit="tokens/s",
)

# Counter for total tokens generated
token_counter = meter.create_counter(
    name="llm.tokens_generated",
    description="Total number of tokens generated",
    unit="1",
)

print("Metrics configured — ready to track TTFT and TPS")


Metrics configured — ready to track TTFT and TPS


## 3. Service Implementations

Each function:
1. Accepts a propagation **carrier** (dict simulating HTTP headers) from the upstream caller.
2. Extracts the parent context from the carrier so its span joins the existing trace.
3. Does its work inside a child span.
4. Injects its own context into a new carrier before calling the next service.

The `session_id` is passed as a span attribute on every span so you can filter the whole trace by session.

## 3a. Enhanced LLM Service with SSE Token Streaming (TTFT & TPS)

The updated `call_llm_service_with_streaming` function:
- Simulates receiving tokens in **SSE (Server-Sent Events)** format  
- Calculates **Time to First Token (TTFT)** and **Tokens Per Second (TPS)**
- Records these metrics to OpenTelemetry
- Sets span attributes for detailed tracing


In [ ]:

import random
import time as time_module

def simulate_sse_token_stream(prompt: str):
    """
    Simulates receiving tokens in SSE format from an LLM API.
    Yields tokens with realistic streaming delays (simulating network/generation delays).
    """
    # Simulate different response delays based on prompt
    base_delay = random.uniform(0.05, 0.15)  # Time before first token
    inter_token_delay = random.uniform(0.01, 0.05)  # Delay between tokens
    
    # Simulate a realistic token response
    tokens = [
        "This", "is", "a", "streaming", "response.", "The", "LLM",
        "generates", "tokens", "one", "at", "a", "time,", "simulating",
        "the", "real", "SSE", "format", "used", "by", "modern", "LLM", "APIs."
    ]
    
    # Add initial delay before first token
    time_module.sleep(base_delay)
    
    for token in tokens:
        yield token
        # Inter-token delay (simulating generation/network latency)
        time_module.sleep(inter_token_delay)


def call_llm_service_with_streaming(carrier: dict, session_id: str, prompt: str) -> str:
    """
    Enhanced LLM service that handles token streaming (SSE format).
    Calculates and reports TTFT (Time to First Token) and TPS (Tokens Per Second) as OTEL metrics.
    """
    tracer = trace.get_tracer("llm-service")
    ctx = propagator.extract(carrier)

    with tracer.start_as_current_span("llm-service.generate_streaming", context=ctx) as span:
        span.set_attribute("session.id", session_id)
        span.set_attribute("service.name", "llm-service")
        span.set_attribute("llm.prompt", prompt)
        span.set_attribute("llm.model", "gpt-demo-streaming-1")
        span.set_attribute("llm.streaming", True)

        # ────────────────────────────────────────────────────────────────────────────
        # Token Collection & Metrics Calculation
        # ────────────────────────────────────────────────────────────────────────────
        
        request_start_time = time_module.time()  # Time when we send the request
        first_token_time = None
        total_tokens = 0
        full_response = ""
        
        # Stream tokens from simulated SSE
        for token in simulate_sse_token_stream(prompt):
            if first_token_time is None:
                # First token received!
                first_token_time = time_module.time()
                ttft_ms = (first_token_time - request_start_time) * 1000
                
                # Record TTFT as a metric attribute
                span.set_attribute("llm.ttft_ms", ttft_ms)
                print(f"    ✓ First token received: TTFT = {ttft_ms:.1f}ms")
            
            full_response += token + " "
            total_tokens += 1
        
        # Calculate TPS (Tokens Per Second)
        completion_time = time_module.time()
        total_duration_seconds = completion_time - request_start_time
        tps = total_tokens / total_duration_seconds if total_duration_seconds > 0 else 0
        
        # ────────────────────────────────────────────────────────────────────────────
        # Record Metrics to OpenTelemetry
        # ────────────────────────────────────────────────────────────────────────────
        
        if first_token_time is not None:
            ttft_ms = (first_token_time - request_start_time) * 1000
            ttft_histogram.record(ttft_ms, {"llm.model": "gpt-demo-streaming-1", "session.id": session_id, "ttft_ms": ttft_ms})
        
        tps_histogram.record(tps, {"llm.model": "gpt-demo-streaming-1", "session.id": session_id, "tps": tps})
        token_counter.add(total_tokens, {"llm.model": "gpt-demo-streaming-1", "session.id": session_id})
        
        # Set span attributes for tracing
        span.set_attribute("llm.tokens_generated", total_tokens)
        span.set_attribute("llm.tps", round(tps, 2))
        span.set_attribute("llm.total_duration_ms", round(total_duration_seconds * 1000, 1))
        span.set_attribute("llm.completion", full_response.strip())
        span.set_status(trace.StatusCode.OK)
        
        print(f"    ✓ Completion: {total_tokens} tokens in {total_duration_seconds:.2f}s ({tps:.2f} tokens/s)")
        print(f"  [llm-service]  trace={span.get_span_context().trace_id:#x}  span={span.get_span_context().span_id:#x}")
        
        return full_response.strip()


print("Enhanced LLM service with streaming and metrics defined.")


Enhanced LLM service with streaming and metrics defined.


In [4]:
def call_llm_service(carrier: dict, session_id: str, prompt: str) -> str:
    """Leaf service — calls an LLM and returns a completion."""
    tracer = trace.get_tracer("llm-service")
    ctx = propagator.extract(carrier)

    with tracer.start_as_current_span("llm-service.generate", context=ctx) as span:
        span.set_attribute("session.id", session_id)
        span.set_attribute("service.name", "llm-service")
        span.set_attribute("llm.prompt", prompt)
        span.set_attribute("llm.model", "gpt-demo-1")

        # Simulate LLM latency
        time.sleep(0.3)
        completion = f"[LLM response to: '{prompt}']"

        span.set_attribute("llm.completion", completion)
        span.set_status(trace.StatusCode.OK)
        print(f"  [llm-service]  trace={span.get_span_context().trace_id:#x}  span={span.get_span_context().span_id:#x}")
        return completion


def call_backend(carrier: dict, session_id: str, request_body: dict) -> dict:
    """Backend — processes business logic, then calls the LLM service."""
    tracer = trace.get_tracer("backend")
    ctx = propagator.extract(carrier)

    with tracer.start_as_current_span("backend.process", context=ctx) as span:
        span.set_attribute("session.id", session_id)
        span.set_attribute("service.name", "backend")
        span.set_attribute("request.user_id", request_body.get("user_id", "unknown"))

        time.sleep(0.05)
        print(f"  [backend]      trace={span.get_span_context().trace_id:#x}  span={span.get_span_context().span_id:#x}")

        # Propagate context downstream to the LLM service
        downstream_carrier: dict = {}
        propagator.inject(downstream_carrier)

        completion = call_llm_service(
            carrier=downstream_carrier,
            session_id=session_id,
            prompt=request_body.get("prompt", ""),
        )

        span.set_status(trace.StatusCode.OK)
        return {"completion": completion, "processed_by": "backend"}


def call_gateway(carrier: dict, session_id: str, request: dict) -> dict:
    """API Gateway — authenticates/routes the request, then calls the backend."""
    tracer = trace.get_tracer("gateway")
    ctx = propagator.extract(carrier)

    with tracer.start_as_current_span("gateway.route", context=ctx) as span:
        span.set_attribute("session.id", session_id)
        span.set_attribute("service.name", "gateway")
        span.set_attribute("http.method", request.get("method", "POST"))
        span.set_attribute("http.route", request.get("route", "/api/chat"))

        time.sleep(0.02)
        print(f"  [gateway]      trace={span.get_span_context().trace_id:#x}  span={span.get_span_context().span_id:#x}")

        downstream_carrier: dict = {}
        propagator.inject(downstream_carrier)

        result = call_backend(
            carrier=downstream_carrier,
            session_id=session_id,
            request_body=request.get("body", {}),
        )

        span.set_attribute("http.status_code", 200)
        span.set_status(trace.StatusCode.OK)
        return result


def call_frontend(session_id: str, user_message: str) -> dict:
    """Frontend — entry point; starts the root span for the entire trace."""
    tracer = trace.get_tracer("frontend")

    with tracer.start_as_current_span("frontend.send_message") as span:
        span.set_attribute("session.id", session_id)
        span.set_attribute("service.name", "frontend")
        span.set_attribute("user.message", user_message)

        print(f"  [frontend]     trace={span.get_span_context().trace_id:#x}  span={span.get_span_context().span_id:#x}")

        # Inject trace context into simulated HTTP headers for the gateway
        downstream_carrier: dict = {}
        propagator.inject(downstream_carrier)

        response = call_gateway(
            carrier=downstream_carrier,
            session_id=session_id,
            request={
                "method": "POST",
                "route": "/api/chat",
                "body": {
                    "user_id": "user-42",
                    "prompt": user_message,
                },
            },
        )

        span.set_status(trace.StatusCode.OK)
        return response

print("Service functions defined.")

Service functions defined.


## 4. Run a Simulated Request

Each run creates a fresh `session_id`. All four spans share the same `trace_id`, forming a single end-to-end trace.

## 4a. Test Streaming LLM Service (with TTFT & TPS Metrics)

In [5]:

# Define a modified backend and gateway that use the streaming LLM service

def call_backend_streaming(carrier: dict, session_id: str, request_body: dict) -> dict:
    """Backend — calls the updated streaming LLM service."""
    tracer = trace.get_tracer("backend")
    ctx = propagator.extract(carrier)

    with tracer.start_as_current_span("backend.process", context=ctx) as span:
        span.set_attribute("session.id", session_id)
        span.set_attribute("service.name", "backend")
        span.set_attribute("request.user_id", request_body.get("user_id", "unknown"))

        time_module.sleep(0.05)
        print(f"  [backend]      trace={span.get_span_context().trace_id:#x}  span={span.get_span_context().span_id:#x}")

        downstream_carrier: dict = {}
        propagator.inject(downstream_carrier)

        completion = call_llm_service_with_streaming(
            carrier=downstream_carrier,
            session_id=session_id,
            prompt=request_body.get("prompt", ""),
        )

        span.set_status(trace.StatusCode.OK)
        return {"completion": completion, "processed_by": "backend-streaming"}


def call_gateway_streaming(carrier: dict, session_id: str, request: dict) -> dict:
    """API Gateway — routes the request with streaming enabled."""
    tracer = trace.get_tracer("gateway")
    ctx = propagator.extract(carrier)

    with tracer.start_as_current_span("gateway.route", context=ctx) as span:
        span.set_attribute("session.id", session_id)
        span.set_attribute("service.name", "gateway")
        span.set_attribute("http.method", request.get("method", "POST"))
        span.set_attribute("http.route", request.get("route", "/api/chat"))

        time_module.sleep(0.02)
        print(f"  [gateway]      trace={span.get_span_context().trace_id:#x}  span={span.get_span_context().span_id:#x}")

        downstream_carrier: dict = {}
        propagator.inject(downstream_carrier)

        result = call_backend_streaming(
            carrier=downstream_carrier,
            session_id=session_id,
            request_body=request.get("body", {}),
        )

        span.set_attribute("http.status_code", 200)
        span.set_status(trace.StatusCode.OK)
        return result


def call_frontend_streaming(session_id: str, user_message: str) -> dict:
    """Frontend — entry point; initiates a trace with streaming LLM."""
    tracer = trace.get_tracer("frontend")

    with tracer.start_as_current_span("frontend.send_message") as span:
        span.set_attribute("session.id", session_id)
        span.set_attribute("service.name", "frontend")
        span.set_attribute("user.message", user_message)

        print(f"  [frontend]     trace={span.get_span_context().trace_id:#x}  span={span.get_span_context().span_id:#x}")

        downstream_carrier: dict = {}
        propagator.inject(downstream_carrier)

        response = call_gateway_streaming(
            carrier=downstream_carrier,
            session_id=session_id,
            request={
                "method": "POST",
                "route": "/api/chat/stream",
                "body": {
                    "user_id": "user-42",
                    "prompt": user_message,
                },
            },
        )

        span.set_status(trace.StatusCode.OK)
        return response


# ── Run a single streaming request ────────────────────────────────────────────
session_id = str(uuid.uuid4())
user_message = "Explain streaming LLM tokens and metrics collection."

print(f"\nSession ID : {session_id}")
print(f"User prompt: {user_message}")
print("─" * 70)

response = call_frontend_streaming(session_id=session_id, user_message=user_message)

print("─" * 70)
print("Response (first 100 chars):", response["completion"][:100] + "...")



Session ID : 56afe17d-b221-4cad-ac45-e5e877b07f32
User prompt: Explain streaming LLM tokens and metrics collection.
──────────────────────────────────────────────────────────────────────
  [frontend]     trace=0xf778e9277c0fc34ea71c496c240082af  span=0x9b7103e5806722dd
  [gateway]      trace=0xf778e9277c0fc34ea71c496c240082af  span=0xf425c69b3a67701b
  [backend]      trace=0xf778e9277c0fc34ea71c496c240082af  span=0xf49115be7450c15
    ✓ First token received: TTFT = 58.9ms
    ✓ Completion: 23 tokens in 0.60s (38.59 tokens/s)
  [llm-service]  trace=0xf778e9277c0fc34ea71c496c240082af  span=0x3d1bdd892d26230b
──────────────────────────────────────────────────────────────────────
Response (first 100 chars): This is a streaming response. The LLM generates tokens one at a time, simulating the real SSE format...


## 4b. Run Multiple Streaming Sessions and Collect Metrics

In [6]:

# Run multiple streaming sessions to gather metrics across different sessions

# Set to True to continuously emit traces + metrics (press CTRL+C to stop)
CONTINUOUS_STREAMING = True

streaming_prompts = [
    "What is Time to First Token (TTFT)?",
    "How do we measure Tokens Per Second (TPS)?",
    "Why is streaming important for LLM APIs?",
    "Describe real-time token monitoring in machine learning.",
]


def run_streaming_sessions(prompts, max_sessions=None, delay_between=0.5):
    """Run a series of streaming sessions.

    If max_sessions is None, runs indefinitely until interrupted.
    """

    session_count = 0
    while True:
        if max_sessions is not None and session_count >= max_sessions:
            break

        prompt = random.choice(prompts)
        sid = str(uuid.uuid4())
        print(f"\n── Streaming Session {session_count+1} ({sid[:8]}…) ─────────────────────────────")
        result = call_frontend_streaming(session_id=sid, user_message=prompt)
        print("Response (first 80 chars):", result["completion"][:80] + "...")

        session_count += 1

        if not CONTINUOUS_STREAMING:
            continue

        time_module.sleep(delay_between)


if CONTINUOUS_STREAMING:
    print("\nRunning streaming sessions continuously (press CTRL+C to stop)")
    try:
        run_streaming_sessions(streaming_prompts, max_sessions=None, delay_between=1.0)
    except KeyboardInterrupt:
        print("\nStopped continuous streaming.")
else:
    print(f"\n{'=' * 70}")
    print("Running 4 streaming sessions with OTEL metrics collection")
    print(f"{'=' * 70}")

    run_streaming_sessions(streaming_prompts, max_sessions=4, delay_between=0.5)

    print(f"\n{'=' * 70}")
    print("✓ All streaming sessions completed")
    print("✓ TTFT and TPS metrics have been recorded to OpenTelemetry")
    print(f"{'=' * 70}")



Running streaming sessions continuously (press CTRL+C to stop)

── Streaming Session 1 (c0834c47…) ─────────────────────────────
  [frontend]     trace=0xdc8e155e952a60f70c432a45a39f861c  span=0x96b3befb9af6b9
  [gateway]      trace=0xdc8e155e952a60f70c432a45a39f861c  span=0x580f9108c1b7000b
  [backend]      trace=0xdc8e155e952a60f70c432a45a39f861c  span=0xcee6ee53e6be067c
    ✓ First token received: TTFT = 76.7ms
    ✓ Completion: 23 tokens in 0.31s (73.16 tokens/s)
  [llm-service]  trace=0xdc8e155e952a60f70c432a45a39f861c  span=0xeaf40c3784a71519
Response (first 80 chars): This is a streaming response. The LLM generates tokens one at a time, simulating...

── Streaming Session 2 (a0d9c9ab…) ─────────────────────────────
  [frontend]     trace=0x305426d52a94364df17edc0cf285b8f6  span=0xf49c2742b1aef3ca
  [gateway]      trace=0x305426d52a94364df17edc0cf285b8f6  span=0x33ae98daabab1c04
  [backend]      trace=0x305426d52a94364df17edc0cf285b8f6  span=0x2585b4220b64850c
    ✓ First token 

{
    "name": "llm-service.generate_streaming",
    "context": {
        "trace_id": "0xb68ab74b7ab2a867c6b68d9b7defcf8f",
        "span_id": "0x2682e63cc9739a0e",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xb84ae16e74d5d5da",
    "start_time": "2026-03-18T06:13:13.660388Z",
    "end_time": "2026-03-18T06:13:14.588324Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "session.id": "42cd7202-eaab-4de9-9b46-604163897da4",
        "service.name": "llm-service",
        "llm.prompt": "How do we measure Tokens Per Second (TPS)?",
        "llm.model": "gpt-demo-streaming-1",
        "llm.streaming": true,
        "llm.ttft_ms": 108.33144187927246,
        "llm.tokens_generated": 23,
        "llm.tps": 24.8,
        "llm.total_duration_ms": 927.5,
        "llm.completion": "This is a streaming response. The LLM generates tokens one at a time, simulating the real SSE format used by modern LLM APIs."
    },
    "events"

session_id = str(uuid.uuid4())
user_message = "Explain the concept of distributed tracing."

print(f"Session ID : {session_id}")
print(f"User prompt: {user_message}")
print("─" * 70)

response = call_frontend(session_id=session_id, user_message=user_message)

print("─" * 70)
print("Response:", response["completion"])

## 5. Run Multiple Sessions

Simulate several independent user sessions, each with its own trace.

prompts = [
    "What is OpenTelemetry?",
    "How does context propagation work?",
    "Describe the W3C TraceContext standard.",
]

for i, prompt in enumerate(prompts, 1):
    sid = str(uuid.uuid4())
    print(f"\n── Session {i} ({sid[:8]}…) ─────────────────────────────────────────")
    result = call_frontend(session_id=sid, user_message=prompt)
    print("Response:", result["completion"])

## 6. Flush & Shutdown

Force all buffered spans to be exported before the notebook kernel stops.

In [ ]:
provider.force_flush(timeout_millis=5000)
provider.shutdown()

# Also flush and shutdown metrics
if hasattr(metric_provider, 'force_flush'):
    print('force flush metric provider')
    metric_provider.force_flush(timeout_millis=5000)
if hasattr(metric_provider, 'shutdown'):
    metric_provider.shutdown()

print("✓ All spans flushed and exporter shut down.")
print("✓ Metrics (TTFT, TPS, Token Count) flushed to OpenTelemetry Collector.")


---
## Collector Quick-Start

If you don't have a collector running yet, the fastest way to get one is via Docker:

```bash
docker run --rm -p 4317:4317 -p 4318:4318 \
  otel/opentelemetry-collector-contrib:latest
```

To also view traces in a UI, add Jaeger:

```bash
# Terminal 1 – Jaeger all-in-one (UI on http://localhost:16686)
docker run --rm -p 16686:16686 -p 14250:14250 jaegertracing/all-in-one:latest

# Terminal 2 – Collector forwarding to Jaeger
docker run --rm -p 4317:4317 \
  -e OTEL_EXPORTER_OTLP_ENDPOINT=http://host.docker.internal:14250 \
  otel/opentelemetry-collector-contrib:latest
```

Or use the `docker-compose.yml` in this directory if one is provided.

**Filtering by session** — in Jaeger's search UI, use the tag filter `session.id=<your-uuid>` to isolate a single end-to-end trace.